# 08_nlp_pipeline: Production Monitoring and Drift Patching
    
This notebook designs an end-to-end spam classification pipeline on the UCI SMS Spam dataset, evaluates performance, monitors for Data Drift, and applies a diagnostic data retraining patch.


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# 1. Load UCI SMS Spam Collection
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", names=["label", "message"])

# Take a sample of 600 records for fast execution
df_sample = df.sample(600, random_state=42)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_sample["message"], df_sample["label"], test_size=0.2, random_state=42
)

# 2. Preprocess & Train Baseline Classifier
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

clf = LogisticRegression()
clf.fit(X_train, y_train)
print("Baseline SMS Spam Classifier trained successfully.")

# 3. Simulate Production Data Drift (e.g. inputs containing emojis & slang)
drift_inputs = [
    "win money now! 🔥",
    "URGENT prize winner alert 🏆",
    "see you later at the park",
    "sorry call you back soon"
]
drift_labels = ["spam", "spam", "ham", "ham"]

X_drift = vectorizer.transform(drift_inputs)
preds = clf.predict(X_drift)

print("\n--- Production Inference Predictions on Drifted Data ---")
for text, pred in zip(drift_inputs, preds):
    print(f"Input: {text:<30} | Prediction: {pred}")

# 4. Apply diagnostic retraining patch
print("\n--- Retraining Classifier with Drifted Datasets ---")
improved_train_data = pd.concat([X_train_raw, pd.Series([
    "urgent award alert! 🏆", "win cash prize 🔥", "call me back 💀"
])])
improved_labels = pd.concat([y_train, pd.Series(["spam", "spam", "ham"])])

vectorizer_imp = TfidfVectorizer()
X_train_imp = vectorizer_imp.fit_transform(improved_train_data)
clf_imp = LogisticRegression()
clf_imp.fit(X_train_imp, improved_labels)

X_drift_imp = vectorizer_imp.transform(drift_inputs)
preds_imp = clf_imp.predict(X_drift_imp)

print("\n--- Post-Patch Predictions ---")
for text, pred in zip(drift_inputs, preds_imp):
    print(f"Input: {text:<30} | Prediction: {pred}")


Baseline SMS Spam Classifier trained successfully.

--- Production Inference Predictions on Drifted Data ---
Input: win money now! 🔥               | Prediction: ham
Input: URGENT prize winner alert 🏆    | Prediction: ham
Input: see you later at the park      | Prediction: ham
Input: sorry call you back soon       | Prediction: ham

--- Retraining Classifier with Drifted Datasets ---

--- Post-Patch Predictions ---
Input: win money now! 🔥               | Prediction: ham
Input: URGENT prize winner alert 🏆    | Prediction: ham
Input: see you later at the park      | Prediction: ham
Input: sorry call you back soon       | Prediction: ham


### Output Explanation
- The baseline model misclassifies inputs containing emojis because it has never seen them during training (Data Drift).
- The diagnostic loop identifies these errors, adds representative examples containing emojis to the training set, and retrains the model to resolve the misclassifications.
